## In this notebook

- Generate situation time data.

In [1]:
import dotenv
import json
import os
from enum import Enum
from typing import List, Tuple
from pathlib import Path

import boto3
import pandas as pd
import requests

from tqdm.notebook import tqdm

In [2]:
dotenv.load_dotenv()

S3_ACCESS_KEY_ID = os.getenv("S3_ACCESS_KEY_ID")
S3_SECRET_ACCESS_KEY = os.getenv("S3_SECRET_ACCESS_KEY")
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")

## Download shift charts

1. Read games data from S3 in order to get list of game ids
2. Download shift charts from API
3. Save shift charts to S3 raw bucket


In [ ]:
URL_SHIFTCHART = "https://api.nhle.com/stats/rest/en/shiftcharts?cayenneExp=gameId="

In [4]:
s3 = boto3.resource(
    "s3",
    aws_access_key_id=S3_ACCESS_KEY_ID,
    aws_secret_access_key=S3_SECRET_ACCESS_KEY
)

In [5]:
class SeasonType(Enum):
    PRESEASON = 1
    REGULAR = 2
    PLAYOFF = 3
    ALLSTAR = 4


def extract_info_from(key: str) -> Tuple[str, str, str]:
    """Extract game_id, season, and season type from a key.

    Parameters:
    -----------
    key : str
        A string representing the unique identifier S3 bucket file.

    Returns:
    --------
    Tuple[str, str, str]
        A tuple containing information about the game_id, season, and season type extracted
        from the provided key.
    """
    game_id = Path(key).stem

    season = game_id[:4]

    season_type_val = int(game_id[5])
    season_type = SeasonType(season_type_val).name.lower()

    return game_id, season, season_type

In [6]:
def download_shift_chart(game_id: str, bucket_name: str, key: str) -> dict:
    """Download shift chart for a given game id.

    Parameters:
    -----------
    game_id: str
        The id of the game to download the shift chart for.

    Returns:
    --------
    dict
        The shift chart for the given game id.
    """
    url = f"{URL_SHIFTCHART}{game_id}"
    response = requests.get(url=url)

    if response.ok:
        data = json.loads(response.text)
        (
            s3
            .Object(bucket_name=bucket_name, key=key)
            .put(Body=(bytes(json.dumps(data).encode("UTF-8"))))
        )


In [7]:
bucket_raw = s3.Bucket("frozen-facts-center-raw")


for obj_raw in tqdm(bucket_raw.objects.all(), total=len(list(bucket_raw.objects.all()))):
    
    # since we need the data only for game ids, skip non-game JSON files and shift charts
    if not obj_raw.key.endswith(".json") or not obj_raw.key.startswith("games/"):
        continue

    # read game id
    game_id, season, season_type = extract_info_from(key=obj_raw.key)

    # download shift chart and save it to raw bucket
    shift_chart_key = f"shift-charts/{season}/{season_type}/{game_id}.json"
    download_shift_chart(game_id=game_id, bucket_name=bucket_raw.name, key=shift_chart_key)

    # break

  0%|          | 0/7268 [00:00<?, ?it/s]

## Results

- Downloaded historic shift charts.